# Adversarial Robustness of Machine Learning-Based Intrusion Detection Systems

This notebook is the local-only version of our project pipeline. It assumes the raw CICIDS2017 CSV files are in the **same folder as this notebook**.

It combines:

- Amogh's adversarial ML pipeline: PyTorch MLP, FGSM, PGD, and transferability tests.
- Our preprocessing pipeline: safer cleaning, stratified split, confusion matrices, classification reports, and saved artifacts.

The notebook does **not** depend on Kaggle paths or pre-cleaned CSV files.

## 1. Project Goal

Machine-learning intrusion detection systems can perform very well on clean benchmark traffic, but they may be vulnerable to adversarial perturbations. In this experiment, we:

1. Load raw local CICIDS2017 CSV files.
2. Clean and sanitize the data into trainable numeric features.
3. Train classical baseline intrusion-detection models.
4. Train a neural-network IDS model.
5. Evaluate all models on clean test data.
6. Generate adversarial test samples using FGSM and PGD against the neural network.
7. Test whether the adversarial examples transfer to Logistic Regression and Random Forest.

The default target is **broad multiclass classification**, which groups raw CICIDS2017 labels into categories such as `Normal Traffic`, `DDoS`, `DoS`, `Port Scanning`, `Brute Force`, `Web Attacks`, and `Bots`.

## 2. Setup and Configuration

Put your downloaded CICIDS2017 CSV files in the same folder as this notebook. The notebook will search only this local folder.

In [ ]:
import json
import random
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Reproducibility
# -----------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# -----------------------------
# Local data settings
# -----------------------------
# This notebook only searches the folder where the notebook is running.
# Do not add Kaggle paths or external paths here.
LOCAL_CSV_PATTERN = "*.csv"

# Files produced by this notebook should not be loaded as raw input data later.
EXCLUDED_CSV_NAMES = {
    "results_summary.csv",
    "processed_cicids2017_local.csv",
}

# Label options:
#   "broad_multiclass" -> groups raw CICIDS labels into broad classes for the project narrative
#   "raw_multiclass"   -> keeps original CICIDS labels, such as DoS Hulk, PortScan, DDoS
#   "binary"           -> BENIGN vs ATTACK
LABEL_MODE = "broad_multiclass"

# Drop classes with fewer than this many rows before stratified splitting.
# Stratified train/test split requires each class to have at least 2 samples.
MIN_SAMPLES_PER_CLASS = 2

# Drop numeric feature columns with too much missingness after coercion.
MAX_MISSING_RATIO = 0.80

# Optional clipping of extreme numeric values after train/test split.
# This is fit on training data only to avoid leakage.
CLIP_EXTREME_VALUES = True
CLIP_LOWER_QUANTILE = 0.001
CLIP_UPPER_QUANTILE = 0.999

# -----------------------------
# Experiment settings
# -----------------------------
TEST_SIZE = 0.20
BATCH_SIZE = 1024
MLP_EPOCHS = 5
MLP_LEARNING_RATE = 1e-3
DROPOUT = 0.30

# To keep classical baselines fast, train Logistic Regression and Random Forest
# on a stratified subset when the training data is very large.
SKLEARN_TRAIN_LIMIT = 300_000

# Adversarial attack settings from Amogh's first pass.
FGSM_EPSILON = 0.05
PGD_EPSILON = 0.05
PGD_ALPHA = 0.01
PGD_STEPS = 10

OUTPUT_DIR = Path("outputs")
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Searching for local CSV files with pattern: {LOCAL_CSV_PATTERN}")

## 3. Data Loading and Cleaning Helpers

This section assumes raw CICIDS2017 CSV files are in the same folder as the notebook.

The cleaning approach is designed to create usable training/testing data while avoiding obvious leakage:

- Column names are stripped and duplicate columns are removed.
- Raw labels are cleaned and optionally grouped into broader attack categories.
- Non-feature metadata columns such as IP addresses, flow IDs, and timestamps are removed.
- Features are coerced to numeric values.
- Infinite values are replaced with missing values.
- All-null, mostly-missing, and constant feature columns are removed.
- Rare classes that cannot support stratified splitting are removed.
- Median imputation, outlier clipping, and scaling are fit only on the training split.

In [ ]:
def find_local_csv_files():
    """Find CSV files in the same folder as the notebook, excluding generated outputs."""
    csv_files = []
    for path in Path(".").glob(LOCAL_CSV_PATTERN):
        if path.is_file() and path.name not in EXCLUDED_CSV_NAMES:
            if not str(path).startswith("outputs"):
                csv_files.append(path)
    return sorted(csv_files)


def read_csv_robust(path):
    """Read a CSV file with a small fallback for encoding issues."""
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin1")


def normalize_column_names(df):
    """Strip whitespace and remove duplicated columns."""
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    return df


def find_label_column(df):
    """Find the CICIDS label column even if whitespace or casing differs."""
    for col in df.columns:
        if col.strip().lower() == "label":
            return col
    raise ValueError("Could not find a 'Label' column. Make sure these are raw CICIDS2017 CSV files.")


def clean_label_text(value):
    """Normalize label text while preserving human-readable class names."""
    value = str(value).strip()
    value = value.replace("\u2013", "-").replace("\u2014", "-")
    value = value.replace("\ufffd", "-")
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_label_key(value):
    """Create a robust key for mapping raw CICIDS labels."""
    value = clean_label_text(value).lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def map_to_broad_attack_type(label):
    """Map raw CICIDS2017 labels to broad project categories."""
    key = normalize_label_key(label)

    if key == "benign":
        return "Normal Traffic"
    if key == "ddos":
        return "DDoS"
    if key == "portscan":
        return "Port Scanning"
    if key.startswith("dos ") or key in {"dos", "heartbleed"}:
        return "DoS"
    if "patator" in key:
        return "Brute Force"
    if key.startswith("web attack") or "web attack" in key:
        return "Web Attacks"
    if key == "bot":
        return "Bots"
    if "infiltration" in key:
        return "Infiltration"

    return "Other Attacks"


def build_target(raw_labels, label_mode=LABEL_MODE):
    """Build the target series according to the configured label mode."""
    cleaned = raw_labels.astype(str).map(clean_label_text)

    if label_mode == "binary":
        return cleaned.map(lambda label: "BENIGN" if normalize_label_key(label) == "benign" else "ATTACK")

    if label_mode == "raw_multiclass":
        return cleaned

    if label_mode == "broad_multiclass":
        return cleaned.map(map_to_broad_attack_type)

    raise ValueError("LABEL_MODE must be 'broad_multiclass', 'raw_multiclass', or 'binary'.")


def get_metadata_columns(columns):
    """Return columns that should not be used as numeric ML features."""
    metadata_names = {
        "flow id",
        "source ip",
        "src ip",
        "destination ip",
        "dst ip",
        "source port",
        "src port",
        "destination port",
        "dst port",
        "timestamp",
        "date",
        "time",
        "label",
    }

    drop_cols = []
    for col in columns:
        normalized = col.strip().lower()
        if normalized in metadata_names:
            drop_cols.append(col)
        elif normalized.startswith("unnamed"):
            drop_cols.append(col)
    return drop_cols


def load_local_raw_cicids2017():
    """Load raw CICIDS2017 CSV files from the notebook folder and perform global sanitation."""
    csv_files = find_local_csv_files()
    if not csv_files:
        raise FileNotFoundError(
            "No CSV files were found in the same folder as this notebook. "
            "Place the raw CICIDS2017 CSV files beside the notebook and rerun this cell."
        )

    print("Found local CSV files:")
    for file in csv_files:
        print(f"  - {file.name}")

    frames = []
    for file in csv_files:
        temp = read_csv_robust(file)
        temp = normalize_column_names(temp)
        label_col = find_label_column(temp)
        if label_col != "Label":
            temp = temp.rename(columns={label_col: "Label"})
        temp["__source_file"] = file.name
        frames.append(temp)

    df = pd.concat(frames, ignore_index=True, sort=False)
    df = normalize_column_names(df)

    print(f"\nCombined raw shape before cleaning: {df.shape}")

    # Remove rows without a usable label.
    df["Label"] = df["Label"].map(clean_label_text)
    df = df[df["Label"].notna()]
    df = df[df["Label"].astype(str).str.len() > 0]

    # Drop duplicate rows after combining all daily files.
    before_dedup = len(df)
    df = df.drop_duplicates()
    print(f"Dropped duplicate rows: {before_dedup - len(df):,}")

    y = build_target(df["Label"], LABEL_MODE)

    # Remove metadata/non-generalizable fields before numeric conversion.
    drop_cols = get_metadata_columns(df.columns)
    drop_cols.append("__source_file")
    drop_cols = sorted(set(drop_cols))

    X = df.drop(columns=drop_cols, errors="ignore")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.apply(pd.to_numeric, errors="coerce")

    # Remove unusable feature columns.
    all_null_cols = X.columns[X.isna().all()].tolist()
    if all_null_cols:
        X = X.drop(columns=all_null_cols)

    missing_ratio = X.isna().mean()
    mostly_missing_cols = missing_ratio[missing_ratio > MAX_MISSING_RATIO].index.tolist()
    if mostly_missing_cols:
        X = X.drop(columns=mostly_missing_cols)

    constant_cols = [col for col in X.columns if X[col].nunique(dropna=True) <= 1]
    if constant_cols:
        X = X.drop(columns=constant_cols)

    # Keep only rows with a target class that has enough examples for stratified splitting.
    class_counts = y.value_counts()
    valid_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
    keep_mask = y.isin(valid_classes)
    dropped_rare = int((~keep_mask).sum())
    X = X.loc[keep_mask].reset_index(drop=True)
    y = y.loc[keep_mask].reset_index(drop=True)

    metadata = {
        "source": "local_raw_csv_files",
        "csv_files": [str(path) for path in csv_files],
        "label_mode": LABEL_MODE,
        "rows_after_cleaning": int(len(X)),
        "num_features_after_cleaning": int(X.shape[1]),
        "dropped_duplicate_rows": int(before_dedup - len(df)),
        "dropped_all_null_columns": all_null_cols,
        "dropped_mostly_missing_columns": mostly_missing_cols,
        "dropped_constant_columns": constant_cols,
        "dropped_metadata_columns": drop_cols,
        "dropped_rare_class_rows": dropped_rare,
        "max_missing_ratio": MAX_MISSING_RATIO,
    }

    print(f"Final feature shape before split: {X.shape}")
    print(f"Final target shape before split: {y.shape}")
    print(f"Dropped rare-class rows: {dropped_rare:,}")

    return X, y, metadata

## 4. Load the Dataset

The class distribution matters a lot for this project. CICIDS2017 is imbalanced, so we track more than accuracy later in the notebook.

In [ ]:
X, y, data_metadata = load_local_raw_cicids2017()
feature_names = X.columns.tolist()

print("Dataset source:", data_metadata["source"])
print("Label mode:", data_metadata["label_mode"])
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

class_counts = y.value_counts().rename_axis("class_label").reset_index(name="count")
class_counts["percentage"] = 100 * class_counts["count"] / class_counts["count"].sum()
display(class_counts)

display(X.head())

## 5. Quick Class-Balance Check

Weighted F1 can look very strong on imbalanced data because large classes dominate the metric. We therefore also report macro F1 and balanced accuracy.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(class_counts["class_label"].astype(str), class_counts["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Class Distribution")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

## 6. Encode Labels, Split Data, Impute, Clip, and Scale Features

We use a stratified split so the train and test sets preserve class proportions.

To avoid test-data leakage:

- Median imputation values are computed only from the training set.
- Extreme-value clipping bounds are computed only from the training set.
- The `StandardScaler` is fit only on the training set.

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y.astype(str))
class_names = label_encoder.classes_.tolist()

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

# Median imputation fit on training data only.
train_medians = X_train_df.median(numeric_only=True)
train_medians = train_medians.fillna(0)
X_train_clean = X_train_df.fillna(train_medians).fillna(0)
X_test_clean = X_test_df.fillna(train_medians).fillna(0)

# Optional outlier clipping fit on training data only.
if CLIP_EXTREME_VALUES:
    lower_bounds = X_train_clean.quantile(CLIP_LOWER_QUANTILE)
    upper_bounds = X_train_clean.quantile(CLIP_UPPER_QUANTILE)
    X_train_clean = X_train_clean.clip(lower=lower_bounds, upper=upper_bounds, axis=1)
    X_test_clean = X_test_clean.clip(lower=lower_bounds, upper=upper_bounds, axis=1)
else:
    lower_bounds = None
    upper_bounds = None

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

print("Classes:")
for index, name in enumerate(class_names):
    print(f"  {index}: {name}")

print("\nTrain shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

train_distribution = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_distribution = pd.Series(y_test).value_counts(normalize=True).sort_index()
split_check = pd.DataFrame({
    "class_name": class_names,
    "train_percentage": train_distribution.values * 100,
    "test_percentage": test_distribution.values * 100,
})
display(split_check)

## 7. Evaluation Helpers

These helpers keep the reporting consistent across models and attack settings.

In [ ]:
results = []


def compute_metrics(y_true, y_pred):
    """Return the main metrics used in the project."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def add_result(model_name, condition, y_true, y_pred, notes=""):
    """Add one model/condition result to the global results list."""
    metrics = compute_metrics(y_true, y_pred)
    row = {
        "model": model_name,
        "condition": condition,
        **metrics,
        "notes": notes,
    }
    results.append(row)
    return row


def show_classification_summary(model_name, condition, y_true, y_pred):
    """Print metrics and a classification report."""
    metrics = compute_metrics(y_true, y_pred)
    print(f"{model_name} - {condition}")
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


def plot_confusion(y_true, y_pred, title):
    """Plot a confusion matrix."""
    labels = np.arange(len(class_names))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    display_obj = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(9, 7))
    display_obj.plot(ax=ax, xticks_rotation=45, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def make_stratified_subset(X_array, y_array, max_samples, random_state=RANDOM_STATE):
    """Return a stratified subset for faster classical model training."""
    if max_samples is None or len(y_array) <= max_samples:
        return X_array, y_array

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=max_samples,
        random_state=random_state,
    )
    subset_indices, _ = next(splitter.split(X_array, y_array))
    return X_array[subset_indices], y_array[subset_indices]

## 8. Train Classical Baseline Models

We keep Logistic Regression and Random Forest because they make the project stronger: we can compare a neural model against non-neural baselines and later test whether adversarial examples transfer across model families.

For consistency with the adversarial samples, both classical baselines are trained on the scaled feature space.

In [ ]:
X_train_sklearn, y_train_sklearn = make_stratified_subset(
    X_train_scaled,
    y_train,
    max_samples=SKLEARN_TRAIN_LIMIT,
)

print("Classical model training subset:", X_train_sklearn.shape)

log_reg = LogisticRegression(
    max_iter=500,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
log_reg.fit(X_train_sklearn, y_train_sklearn)

random_forest = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_forest.fit(X_train_sklearn, y_train_sklearn)

sklearn_models = {
    "Logistic Regression": log_reg,
    "Random Forest": random_forest,
}

for model_name, model in sklearn_models.items():
    clean_preds = model.predict(X_test_scaled)
    add_result(model_name, "clean", y_test, clean_preds)
    show_classification_summary(model_name, "clean", y_test, clean_preds)
    plot_confusion(y_test, clean_preds, f"{model_name} - Clean Test Data")

## 9. Train the Neural Network IDS Model

This section keeps Amogh's PyTorch MLP idea, because it gives us a differentiable model for FGSM and PGD attacks.

Architecture:

- Input: one neuron per network-flow feature
- Hidden layer: 128 neurons + ReLU + dropout
- Hidden layer: 64 neurons + ReLU
- Output: one neuron per target class

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=DROPOUT):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.model(x)


X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

mlp = MLP(input_dim=X_train_scaled.shape[1], num_classes=len(class_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=MLP_LEARNING_RATE)

print(mlp)

for epoch in range(MLP_EPOCHS):
    mlp.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        outputs = mlp(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{MLP_EPOCHS} - loss: {average_loss:.4f}")

In [ ]:
def predict_mlp(model, data_loader):
    """Return predictions and labels for a PyTorch model/data loader."""
    model.eval()
    predictions = []
    labels = []

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(DEVICE)
            outputs = model(batch_X)
            batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(batch_preds)
            labels.extend(batch_y.numpy())

    return np.array(labels), np.array(predictions)


mlp_clean_labels, mlp_clean_preds = predict_mlp(mlp, test_loader)
add_result("MLP", "clean", mlp_clean_labels, mlp_clean_preds)
show_classification_summary("MLP", "clean", mlp_clean_labels, mlp_clean_preds)
plot_confusion(mlp_clean_labels, mlp_clean_preds, "MLP - Clean Test Data")

## 10. Generate FGSM and PGD Adversarial Examples

FGSM and PGD perturb the scaled tabular feature vectors in the direction that increases the neural network's loss.

Important limitation: these attacks show model sensitivity in feature space, but they do not guarantee that every perturbed vector corresponds to a physically valid network flow. We should mention this limitation in the final report.

In [ ]:
feature_min = torch.tensor(X_train_scaled.min(axis=0), dtype=torch.float32, device=DEVICE)
feature_max = torch.tensor(X_train_scaled.max(axis=0), dtype=torch.float32, device=DEVICE)


def fgsm_attack(model, X_batch, y_batch, epsilon):
    """Fast Gradient Sign Method attack on scaled feature vectors."""
    model.eval()
    X_adv = X_batch.clone().detach().to(DEVICE)
    y_batch = y_batch.to(DEVICE)
    X_adv.requires_grad = True

    outputs = model(X_adv)
    loss = criterion(outputs, y_batch)

    model.zero_grad()
    loss.backward()

    perturbation = epsilon * X_adv.grad.sign()
    X_adv = X_adv + perturbation
    X_adv = torch.max(torch.min(X_adv, feature_max), feature_min)
    return X_adv.detach()


def pgd_attack(model, X_batch, y_batch, epsilon, alpha, steps):
    """Projected Gradient Descent attack on scaled feature vectors."""
    model.eval()
    X_original = X_batch.clone().detach().to(DEVICE)
    y_batch = y_batch.to(DEVICE)
    X_adv = X_original.clone().detach()

    for _ in range(steps):
        X_adv.requires_grad = True
        outputs = model(X_adv)
        loss = criterion(outputs, y_batch)

        model.zero_grad()
        loss.backward()

        X_adv = X_adv + alpha * X_adv.grad.sign()
        perturbation = torch.clamp(X_adv - X_original, min=-epsilon, max=epsilon)
        X_adv = X_original + perturbation
        X_adv = torch.max(torch.min(X_adv, feature_max), feature_min).detach()

    return X_adv


def generate_adversarial_dataset(model, data_loader, attack_name, **attack_kwargs):
    """Generate adversarial examples and evaluate the MLP on them."""
    model.eval()
    adv_batches = []
    label_batches = []
    predictions = []
    labels = []

    for batch_X, batch_y in data_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        if attack_name == "FGSM":
            adv_X = fgsm_attack(model, batch_X, batch_y, **attack_kwargs)
        elif attack_name == "PGD":
            adv_X = pgd_attack(model, batch_X, batch_y, **attack_kwargs)
        else:
            raise ValueError(f"Unknown attack: {attack_name}")

        adv_batches.append(adv_X.cpu())
        label_batches.append(batch_y.cpu())

        with torch.no_grad():
            outputs = model(adv_X)
            batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()

        predictions.extend(batch_preds)
        labels.extend(batch_y.cpu().numpy())

    X_adv = torch.cat(adv_batches).numpy()
    y_adv = torch.cat(label_batches).numpy()
    return X_adv, y_adv, np.array(labels), np.array(predictions)

In [ ]:
X_test_fgsm, y_test_fgsm, fgsm_labels, fgsm_preds = generate_adversarial_dataset(
    mlp,
    test_loader,
    attack_name="FGSM",
    epsilon=FGSM_EPSILON,
)

add_result("MLP", "FGSM", fgsm_labels, fgsm_preds, notes=f"epsilon={FGSM_EPSILON}")
show_classification_summary("MLP", f"FGSM epsilon={FGSM_EPSILON}", fgsm_labels, fgsm_preds)
plot_confusion(fgsm_labels, fgsm_preds, "MLP - FGSM Adversarial Test Data")

print("X_test_fgsm shape:", X_test_fgsm.shape)

In [ ]:
X_test_pgd, y_test_pgd, pgd_labels, pgd_preds = generate_adversarial_dataset(
    mlp,
    test_loader,
    attack_name="PGD",
    epsilon=PGD_EPSILON,
    alpha=PGD_ALPHA,
    steps=PGD_STEPS,
)

add_result(
    "MLP",
    "PGD",
    pgd_labels,
    pgd_preds,
    notes=f"epsilon={PGD_EPSILON}, alpha={PGD_ALPHA}, steps={PGD_STEPS}",
)
show_classification_summary("MLP", f"PGD epsilon={PGD_EPSILON}", pgd_labels, pgd_preds)
plot_confusion(pgd_labels, pgd_preds, "MLP - PGD Adversarial Test Data")

print("X_test_pgd shape:", X_test_pgd.shape)

## 11. Test Transferability to Classical Models

Here we reuse the FGSM and PGD examples generated against the MLP and evaluate the classical models on those same perturbed feature vectors.

If Logistic Regression or Random Forest performance drops, that suggests the adversarial examples are at least partially transferable across model types.

In [ ]:
adversarial_sets = {
    "FGSM": (X_test_fgsm, y_test_fgsm),
    "PGD": (X_test_pgd, y_test_pgd),
}

for model_name, model in sklearn_models.items():
    for attack_name, (X_adv, y_adv) in adversarial_sets.items():
        adv_preds = model.predict(X_adv)
        add_result(model_name, attack_name, y_adv, adv_preds, notes="transfer attack generated from MLP")
        show_classification_summary(model_name, attack_name, y_adv, adv_preds)

## 12. Results Summary

This table is the main output of the experiment. For the final report, the most important comparison is clean performance versus adversarial performance.

In [ ]:
results_df = pd.DataFrame(results)
metric_cols = ["accuracy", "weighted_f1", "macro_f1", "balanced_accuracy"]
results_display = results_df.copy()
for col in metric_cols:
    results_display[col] = results_display[col].map(lambda value: f"{value:.4f}")

display(results_display)

results_df.to_csv(OUTPUT_DIR / "results_summary.csv", index=False)
with open(OUTPUT_DIR / "results_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved results to {OUTPUT_DIR / 'results_summary.csv'}")
print(f"Saved results to {OUTPUT_DIR / 'results_summary.json'}")

plt.figure(figsize=(10, 5))
plot_df = results_df.copy()
plot_df["label"] = plot_df["model"] + " - " + plot_df["condition"]
plt.bar(plot_df["label"], plot_df["accuracy"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Accuracy")
plt.title("Clean vs. Adversarial Accuracy by Model")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 13. Save Reproducibility Artifacts

This section saves the scaler, imputation values, label encoder, model files, feature list, and experiment configuration. These are necessary if we want to reload the trained models later without rerunning the full notebook.

In [ ]:
# Scikit-learn and preprocessing artifacts
joblib.dump(scaler, MODEL_DIR / "scaler.joblib")
joblib.dump(label_encoder, MODEL_DIR / "label_encoder.joblib")
joblib.dump(train_medians, MODEL_DIR / "train_medians.joblib")
joblib.dump(log_reg, MODEL_DIR / "logistic_regression_model.joblib")
joblib.dump(random_forest, MODEL_DIR / "random_forest_model.joblib")

if CLIP_EXTREME_VALUES:
    joblib.dump(lower_bounds, MODEL_DIR / "clip_lower_bounds.joblib")
    joblib.dump(upper_bounds, MODEL_DIR / "clip_upper_bounds.joblib")

# PyTorch model state
mlp_artifact = {
    "state_dict": mlp.state_dict(),
    "input_dim": X_train_scaled.shape[1],
    "num_classes": len(class_names),
    "class_names": class_names,
    "dropout": DROPOUT,
}
torch.save(mlp_artifact, MODEL_DIR / "mlp_model.pth")

# Feature/config metadata
with open(OUTPUT_DIR / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

experiment_config = {
    "random_state": RANDOM_STATE,
    "data_metadata": data_metadata,
    "label_mode": LABEL_MODE,
    "test_size": TEST_SIZE,
    "batch_size": BATCH_SIZE,
    "mlp_epochs": MLP_EPOCHS,
    "mlp_learning_rate": MLP_LEARNING_RATE,
    "dropout": DROPOUT,
    "sklearn_train_limit": SKLEARN_TRAIN_LIMIT,
    "fgsm_epsilon": FGSM_EPSILON,
    "pgd_epsilon": PGD_EPSILON,
    "pgd_alpha": PGD_ALPHA,
    "pgd_steps": PGD_STEPS,
    "clip_extreme_values": CLIP_EXTREME_VALUES,
    "clip_lower_quantile": CLIP_LOWER_QUANTILE,
    "clip_upper_quantile": CLIP_UPPER_QUANTILE,
    "max_missing_ratio": MAX_MISSING_RATIO,
    "class_names": class_names,
}
with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print(f"Saved artifacts under: {OUTPUT_DIR.resolve()}")

## 14. Current Project Status and Next Steps

At this point, the project has a working experimental backbone:

- Local raw-data loading.
- Clean-data intrusion detection baselines.
- A PyTorch neural-network IDS model.
- FGSM and PGD adversarial evaluation.
- Transferability testing against Logistic Regression and Random Forest.
- Saved results and model artifacts.

Recommended next steps after this cleanup:

1. Add an epsilon sweep to show how attack strength changes accuracy/F1.
2. Add stronger minority-class analysis, especially macro F1 and recall per attack class.
3. Add at least one defense, such as adversarial training.
4. Move reusable functions into `src/` after the notebook is stable.